In [ ]:
"""
    gremban_lift(L::AbstractMatrix)

Constructs the Gremban expansion matrix M from a given matrix L.

# Arguments
- `L::AbstractMatrix`: The input matrix (usually a Laplacian).

# Returns
- `Matrix{Float64}`: The 2n x 2n Gremban expansion matrix.
"""
function gremban_lift(L::AbstractMatrix)
    # Get the size of the matrix (number of rows)
    n = size(L, 1)

    # Initialize a 2n x 2n matrix of zeros as Float64
    M = zeros(Float64, 2n, 2n)

    for i in 1:n
        Off_set_diag = 0.0

        for j in 1:n
            if j == i
                continue
            end

            val = L[i, j]

            if val > 0
                # "Bad" edges (positive off-diagonals) move to the off-diagonal blocks (A)
                # We negate the value as per the standard Gremban reduction
                M[i, n + j] = -val
                M[j, n + i] = -val
                M[n + i, j] = -val
                M[n + j, i] = -val
                Off_set_diag += val
            else
                # "Good" edges (negative off-diagonals) stay in the diagonal blocks
                M[i, j] = val
                M[j, i] = val
                M[i + n, j + n] = val
                M[j + n, i + n] = val
                Off_set_diag += -val
                println(" sum is $Off_set_diag, val is $(-val)")
            end
        end

        diff = L[i, i] - Off_set_diag
        M[i, i] = Off_set_diag + diff / 2
        M[i + n, i + n] = M[i, i]
        M[i, i + n] = -diff / 2
        M[i + n, i] = -diff / 2
    end

    return M
end

In [ ]:
def gremban_lift_hermitian(L,N_max):
    """
    Constructs the Gremban expansion matrix M from a given matrix L.

    Args:
        L (numpy.ndarray): The input matrix (usually a Laplacian).
        N_max: Size of the lift, depends on the

    Returns:
        numpy.ndarray: The 2n x 2n Gremban expansion matrix.
    """
    # Ensure input is a NumPy array
    L = np.array(L)

    # Get the size of the matrix (number of rows)
    n = L.shape[0]

    # Initialize a 2n x 2n matrix of zeros
    M = np.zeros((N_max * n, N_max * n), dtype=float)

    for i in range(n):
        # Set diagonal elements for both diagonal blocks
        # Julia: M[i,i] and M[i+n,i+n]

        #M[i, i] = L[i, i]
        #M[i + n, i + n] = L[i, i]

        # Loop through the upper triangle (excluding diagonal)
        # Julia: for j in i+1:n
        Off_set_diag= 0
        for j in range(n):
            if j==i:
                continue
            val = -L[i, j]
            for k in range(N_max):
               # print(np.exp(1j*2 * np.pi * k/N_max)*val)
               #if (k==3 and i==1 and j==3) or (k==18 and i==1 and j==3) :
                #print(f"current multiple is {np.exp(1j*2 * np.pi * k/N_max)*val}")
                #print(f"the current gap is {np.abs(val)-np.exp(1j*2 * np.pi * k/N_max)*val} and k is {3}")
               if np.isclose(np.exp(-1j*2 * np.pi * k/N_max)*val,np.abs(val),atol=1e-4):
                    Off_set_diag += np.abs(val)
                    for l in range(N_max):
                     #   print(f"block index {k+l%N_max} and small index {j}")
                        M[n*l+i,n*((k+l)%N_max)+j]= -np.abs(val)

        #print(f" round {i}, off is {Off_set_diag}")
        diff = L[i,i]-Off_set_diag
        for k in range(N_max):
            #print(f"accessing {n*k} with current value {M[n*k+i,n*k+i]}")
            M[n*k+i,n*k+i]=Off_set_diag+ diff/2
            #print(f"changed {n*k+i} with current value {M[n*k+i,n*k+i]}")
            #print(f"block index {((int(N_max/2)+k)%N_max)} and small index {i} and runner {k} and step size {int(N_max/2)}")
            M[i+k*n,i+((int(N_max/2)+k)%N_max) *n] =    -diff/2
    return M

In [ ]:
def deconstruct_l_matr(l_to_decon,N,a,b,k):
    l_size= l_to_decon.shape[0]
    l_tilde_size= int(l_size*N)
    matrix_deconstruct= np.zeros([l_tilde_size,N],dtype=complex)
    for i in range(N):
        b_i= np.zeros(l_tilde_size,dtype=complex)
        b_i[i*l_size+a]= 1
        #print(f"index {i} postion{(i+k)%N*l_size}")
        b_i[(i+k)%N*l_size+b]=-1
        matrix_deconstruct[:,i] = b_i
    return matrix_deconstruct

In [ ]:
def eff_ref_and_trans_imb_matr(lift,l_new,N_max,a,b,phi_k):
    pinv_lift= np.linalg.pinv(lift)
    b_matrix= deconstruct_l_matr(l_new,N_max,a,b,phi_k)
    print(b_matrix.shape)
    zero_total =0
    total_eff_ref=0
    trans_tot_abs=0
    r_eff_matrix= np.transpose(b_matrix)@pinv_lift@ b_matrix
    #print(r_eff_matrix[0,1])
    for i in range(b_matrix.shape[1]):
        #print(np.sum(np.abs(r_eff_matrix[:,i]))-r_eff_matrix[i,i])
        for j in range(i):
            trans_imb= r_eff_matrix[i,j]
            trans_tot_abs+= 2* np.abs(trans_imb)
            zero_total+= 2 * np.cos(2*np.pi *(j-i)/N_max)* trans_imb
    #print(r_eff_matrix[0,0])
    total_eff_ref= np.sum(r_eff_matrix.diagonal())
    #print(f"total sum is  {np.sum(np.abs(r_eff_matrix))}")
    print(f"zero total impedenance is  {zero_total}")
    #print(f"total trans imbedance absolute is {trans_tot_abs}")
    print(f"total eff ref is {total_eff_ref}")
    return total_eff_ref,zero_total

In [ ]:
function example_hdd(n,k)
  theta= 2*pi/k
  a=ones(ComplexF64, n, n)
  S= n *diagm(ones(ComplexF64, n))-a
  S[1,2]= -exp(-1im * theta)
  S[2,1]= -exp(1im * theta)
  return S
end

In [ ]:
n = 2000
for j=1:10
    k = 2^j
    S_k = example_hdd(n,k)
    l = zeros(n)
    l[3] = 1
    l[4] = 1
    println(1/(5*k) * l' * pinv(S_k) * l)
    println(minimum(eigvals(S_k)))
end